# ravensML Tutorial

## Description

ravensML provides a suite of tools designed to facilitate the use of PyTorch machine learning capabilities on MG-RAVENS grid data. The framework offers a streamlined pipeline that converts MG-RAVENS files into accessible RavensData Python data objects, enabling a wide range of custom analysis and processing methods. Additionally, ravensML includes functionality to transform these intermediate representations into PyTorch-compatible datasets. This comprehensive toolset bridges the gap between power grid data and modern machine learning techniques, allowing researchers and engineers to apply sophisticated analytical approaches to grid problems. The framework particularly shines in graph-based applications, as demonstrated in our example workflow where a Graph Neural Network (GNN) is used to predict transformer parameters from MG-RAVENS grid data. By removing technical barriers between domain-specific grid representations and machine learning libraries, ravensML empowers users to develop and deploy advanced ML solutions for power systems analysis with minimal overhead.

In this example we will implement a Graph Neural Network to predict shortfall in power-flow given a distribution feeder provided in the MG-RAVENs data schema. This will require us to define a dataset class, define the model structure, select an appropriate loss function, and then implement a standard PyTorch training loop. 

## Prerequisites
- Basic Python knowledge
- Familiarity with PyTorch (helpful but not required)
- Understanding of power grid concepts (recommended)

## What You'll Learn
- Convert MG-RAVENS data to PyTorch datasets
- Build a custom GNN for grid analysis
- Train and evaluate models on power systems data

## Imports/Setup

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import BatchNorm, PNAConv
from torch_geometric.loader import DataLoader
from torch_geometric.utils import degree
from ravens.ravensML.framework.templates.t_pyg_dataset import t_MG_Dataset
from ravens.ravensML.framework.tools.training_tools import train_epoch, validate

/Users/oreed/Desktop/LANL-ANSI/MG-RAVENS/testing_ravens/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
rML_ROOT = Path(os.getcwd()).parents[0]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Tutorial Case

### Dataset

#### Premise: 

ravensML provides a template class implementing many of the core methods needed to convert ravensML data directly to a PyTorch dataset. All that is left to the user is to implement the `_get_target()` method and optionally implement the `_apply_synth_transform()` method.

`_get_target()` is the method through which the user designates the associated `y` value to serve as the target being predicted by the ML method for every input grid.

`_apply_synth_transform()` allows the user to generate a series of synthetic datapoints given the provided directory of MG-RAVENS files yielding the following data pipeline.

$$
X \rightarrow [\text{Optional: }X_{synth} ] \rightarrow \text{NN} \rightarrow y
$$


In this case, `get_target()` pulls the pre-calculated network power flow shortfall and sets it as the Y value for datapoint `i` without need for further calculation. 

`_apply_synth_transform()` calls `inf_data_gen()`, which performs the following operations:

1. Selects `target_size` files from the input `mgr` dataset without replacement
2. Applies Gaussian noise with mean and standard deviation specified at instantiation through `error_kwargs = {"mean": 0.0, "std": 0.5}`
3. Runs a power flow computation through PowerModelsDistribution
4. Stores the results

#### Implementation:

In [ ]:
class MG_Inf_Dataset(t_MG_Dataset):
    def _get_target(self, i, mgr_path, corrupt_dict, corrupt_data, aux):
        Y_inf = aux["infeasibility"]
        return Y_inf[i]
    
    def _apply_synth_transform(self,mgr,target_size):
        from framework.tools.pf_inf_approx.inf_data_gen import inf_data_gen
        (X_mgr, Y_inf) = inf_data_gen(
            mgr,
            size = target_size,
            seed=0,
            **self.synth_kwargs, #provided in the template class to guide synthetic data generation
        )
        X_mgr.process_for_ML()
        aux = {"infeasibility": Y_inf} #stores the results of the transform in an auxiliary data dictionary.
        return X_mgr, aux

### Model

#### Premise:

Within the ravensML framework implementing a PyTorch Graph Neural Network for MG-RAVENS data is no different than any other TorchGeometric implementation. Provided below is an implementation of a PNA convolution based method.

On every forward pass the SimpleGNN model preforms `transport_distance` PNA convolutions before feeding the output into an MLP head that outputs the final power flow infeasibility value. 

#### Implementation:

In [5]:
class SimpleGNN(nn.Module):
    """
    Parameters
    ----------
    node_features : int
        Dimensionality of node feature vectors.
    edge_features : int
        Dimensionality of edge feature vectors.
    degree : int
        Maximum node degree in the training graphs (required by PNA).
    max_nodes : int
        Upper bound on the number of nodes a graph can have.
    transport_distance : int, default 5
        Number of successive PNA message-passing steps.
    """

    def __init__(
        self,
        node_features: int,
        edge_features: int,
        degree,
        max_nodes,
        transport_distance: int = 5,
    ):
        super().__init__()

        # Cast to plain Python ints – safeguards against tensors/np scalars.
        self.degree = degree
        self.max_nodes = int(max_nodes)

        aggregators = ["mean", "min", "max", "std"]
        scalers = ["identity", "amplification", "attenuation"]

        # PNA graph convolutions
        self.convs = nn.ModuleList(
            [
                PNAConv(
                    node_features,
                    node_features,
                    aggregators=aggregators,
                    scalers=scalers,
                    deg=self.degree,
                    edge_dim=edge_features,
                    towers=5,
                    pre_layers=1,
                    post_layers=1,
                    divide_input=False,
                )
                for _ in range(transport_distance)
            ]
        )
        self.norms = nn.ModuleList(
            [BatchNorm(node_features) for _ in range(transport_distance)]
        )

        # Edge-wise MLP – final head outputs `max_nodes ** 2` logits.
        self.edge_mlp = nn.Sequential(
            nn.Linear(node_features * 2 + edge_features, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.Dropout(0.21),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.Dropout(0.20),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.Dropout(0.19),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.Dropout(0.18),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.Dropout(0.17),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.Dropout(0.17),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.Dropout(0.17),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.Dropout(0.16),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.Dropout(0.15),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, data):
        """
        Returns
        -------
        adj : Tensor of shape (max_nodes, max_nodes, params)
              Values are squeezed into [0, 1] (no soft-max).
        """
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Graph convolutions
        for conv, bn in zip(self.convs, self.norms):
            x = F.relu(bn(conv(x, edge_index, edge_attr)))

        # Edge representation: concat(src_node, dst_node, edge_attr)
        src = x[edge_index[0]]
        dst = x[edge_index[1]]
        edge_rep = torch.cat([src, dst, edge_attr], dim=-1)   # (E, 2*F + edge_features)

        # Deep MLP --> (E, max_nodes^2) logits
        edge_logits = self.edge_mlp(edge_rep)                # (E, max_nodes^2)

        # Aggregate over edges --> a single value per graph.
        adj = edge_logits.mean(dim=0)

        return adj

### Training Loop

#### Premise:

The goal of ravensML is to allow for the implementation of a completely boilerplate PyTorch ML workflow that  abstracts out the need to interface with the underlying MG-RAVENS grid objects. This is meant to make it far easier for those who are not experts on the MG-RAVENs format, programming, ML, and electrical engineering to rapidly develop test and deploy advanced ML methods enabling advanced power systems analysis. 

The core components of the training loop are:
- Dataset and Dataloader instantiation
- Model instantiation
- Iterative forward and backward training passes encapsulated by the provided `train_epoch` and `validate` functions

#### Implementation:

In [6]:
dataset = MG_Inf_Dataset(
    root=rML_ROOT,
    path = "data/seg_data",
    size=100, #NOTE: Extremely small sample size for the sake of this example
    error_kwargs={"mean": 0.25,"std": 1.25},
)


# split
train_len = int(0.8 * len(dataset))
val_len   = len(dataset) - train_len
train_set, val_set = torch.utils.data.random_split(dataset, [train_len, val_len])

# data loaders
batch_size = 50 #NOTE: also should be scaled with size
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)


/Users/oreed/Desktop/LANL-ANSI/MG-RAVENS/testing_ravens/lib/python3.13/site-packages/julia/juliainfo.py:93: UserWarning: /Users/oreed/.juliaup/bin/julia warned:
The latest version of Julia in the `release` channel is 1.12.6+0.aarch64.apple.darwin14. You currently have `1.11.6+0.aarch64.apple.darwin14` installed. Run:

  juliaup update

in your terminal shell to install Julia 1.12.6+0.aarch64.apple.darwin14 and update the `release` channel to that version.
  warnings.warn("{} warned:\n{}".format(julia, stderr))



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************



In [7]:
# input / output dimensions 
sample = dataset[0]
node_feat_dim = sample.x.shape[1]
edge_feat_dim = sample.edge_attr.shape[1]


# Compute the maximum in-degree in the training data.
max_degree = -1
for data in train_set:
    d = degree(data.edge_index[1], num_nodes=data.num_nodes, dtype=torch.long)
    max_degree = max(max_degree, int(d.max()))

# Compute the in-degree histogram tensor
deg = torch.zeros(max_degree + 1, dtype=torch.long)
for data in train_set:
    d = degree(data.edge_index[1], num_nodes=data.num_nodes, dtype=torch.long)
    deg += torch.bincount(d, minlength=deg.numel())


# model
model = SimpleGNN(                 
    node_features=node_feat_dim,
    edge_features=edge_feat_dim,
    degree=deg,
    max_nodes = 20,
    transport_distance=12
).to(device)

In [ ]:
# Training Settings
loss_fn = nn.MSELoss() 
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
epochs = 20 #NOTE: should scale with real example

# training parameters
best_val = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, epochs + 1):
    tr_loss = train_epoch(model, train_loader, loss_fn, optimizer, device, 20)
    va_loss = validate(model, val_loader, loss_fn, device, 20)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    scheduler.step(va_loss)

    print(f"Epoch {epoch:02d}/{epochs} | "
            f"Train MSE: {tr_loss:.6f} | Val MSE: {va_loss:.6f}")

    # checkpoint
    if va_loss < best_val:
        best_val = va_loss
        torch.save(model.state_dict(), rML_ROOT/f"documentation/tmp/tutorial_best_model.pth")
        print("  -> saved new best model")

Epoch 01/20 | Train MSE: 93024.168039 | Val MSE: 2765.987793
  -> saved new best model


Epoch 02/20 | Train MSE: 90660.267020 | Val MSE: 8916.885742


Epoch 03/20 | Train MSE: 92419.928013 | Val MSE: 13116.848633


Epoch 04/20 | Train MSE: 92909.196429 | Val MSE: 7772.424316


Epoch 05/20 | Train MSE: 91175.926583 | Val MSE: 3784.506104


...

Epoch 16/20 | Train MSE: 91470.799258 | Val MSE: 1400.104492


Epoch 17/20 | Train MSE: 91398.200250 | Val MSE: 1536.184814


Epoch 18/20 | Train MSE: 91479.272252 | Val MSE: 1823.413696


Epoch 19/20 | Train MSE: 91277.378745 | Val MSE: 2456.462158


Epoch 20/20 | Train MSE: 91163.463867 | Val MSE: 2979.950439
